In [2]:
import requests
import tomllib
from pathlib import Path

secrets = Path("../secrets")
with open(secrets/"lastfm.toml", "rb") as f:
    lastfm_toml = tomllib.load(f)
client_id = lastfm_toml.get("lastfm-credentials").get("client_id")

In [3]:
BASE_URL = "http://ws.audioscrobbler.com/2.0/"
USER = "GammelPerson"
HEADER = {"user-agent": "GammelPerson_2025_data"}
request_parameters = {
        "api_key": client_id,
        "method": "artist.getInfo",
        "artist": "Phil Wangseyt",
        "autocorrect": 1,
        "format": "json"
    }

In [4]:
response = requests.get(url=BASE_URL, params=request_parameters, headers=HEADER)

In [5]:
if not response.raise_for_status():
    out = response.json()

out

{'error': 6,
 'message': 'The artist you supplied could not be found',
 'links': []}

In [6]:
def extract_track_data(response:dict):
    out_dict = {}
    track_data = response.get("track")

    # track info
    out_dict["track_name"] = track_data.get("name")
    out_dict["track_mbid"] = track_data.get("mbid")
    out_dict["track_duration"] = track_data.get("duration")

    out_dict["artist"] = track_data.get("artist").get("name")

    # last 
    out_dict["listeners"] = track_data.get("listeners")
    out_dict["playcount"] = track_data.get("playcount")

    album_data = track_data.get("album")
    out_dict["album_title"] = album_data.get("title")
    out_dict["album_mbid"] = album_data.get("mbid")
    out_dict["album_image_url"] = album_data.get("image")[2].get("#text")

    toptags = track_data.get("toptags").get("tag")
    out_dict["genre_tags"] = [tag.get("name") for tag in toptags] if toptags else None
    return out_dict

In [7]:
import pendulum
from math import floor
def extract_artist_data(response:dict):
    out_dict = {}
    artist_data = response.get("artist")

    out_dict["artist"] = artist_data.get("name")
    out_dict["artist_mbid"] = artist_data.get("mbid")
    
    out_dict["artist_image_url"] = artist_data.get("image")[2].get("#text")

    # last.fm stats
    out_dict["listeners"] = artist_data.get("stats").get("listeners")
    out_dict["playcount"] = artist_data.get("stats").get("playcount")

    # tags
    toptags = artist_data.get("tags").get("tag")
    out_dict["artist_genre_tags"] = [tag.get("name") for tag in toptags] if toptags else None

    # data retrieved
    out_dict["unix_timestamp"] = floor(pendulum.now(tz="UTC").timestamp())

    return out_dict

extract_artist_data(out)

AttributeError: 'NoneType' object has no attribute 'get'

In [8]:
import polars as pl
def get_track_table():
    cols = [
        "date_played_unix",
        "track_played_utc",
        "track_name",
        "artist_name",
        "track_mbid",
    ]
    lastfm_df = pl.read_parquet(
        Path("../data/lastfm/lastfm-listening-2021-2026march.parquet"),
        columns=cols
    )
    spotify_df = pl.read_parquet(
        Path("../data/spotify/2017-2021 spotify data.parquet"),
    )
    # lastfm_df = lastfm_df.with_columns(spotify_track_uri=pl.lit(""))
    # spotify_df = spotify_df.with_columns(
    #     artist_mbid=pl.lit(""), album_mbid=pl.lit(""), track_mbid=pl.lit("")
    # )
    
    # lastfm_df = lastfm_df.select(cols)
    # spotify_df = spotify_df.select(cols)

    # df = lastfm_df.vstack(spotify_df).sort(pl.col("date_played_unix"), descending=False)
    df = pl.concat([ lastfm_df, spotify_df ], how="diagonal").sort(pl.col("date_played_unix"), descending=False)
    df = ( 
        df
        .filter(pl.col("track_played_utc").dt.year() >= 2021)
        .unique(subset=["track_name"], keep="first")
        .sort("track_played_utc", descending=False)
        .drop_nulls(pl.col("track_name"))
    )

    return df

df = get_track_table()

In [11]:
df.filter(
    pl.col("track_played_utc").dt.year() == 2025,
    pl.col("track_played_utc").dt.month() == 2    
)

date_played_unix,track_played_utc,track_name,artist_name,track_mbid,album_name,spotify_track_uri
i64,datetime[μs],str,str,str,str,str
1738433778,2025-02-01 18:16:18,"""Grey skies""","""Bassvictim""","""""",null,null
1738433876,2025-02-01 18:17:56,"""50 / Everything Is Fine""","""Ngahere Wafer""","""""",null,null
1738433986,2025-02-01 18:19:46,"""Forever salty""","""Bassvictim""","""""",null,null
1738434192,2025-02-01 18:23:12,"""Life so far""","""Bassvictim""","""""",null,null
1738434337,2025-02-01 18:25:37,"""I'll be there""","""Bassvictim""","""""",null,null
…,…,…,…,…,…,…
1740753652,2025-02-28 14:40:52,"""Dream Sequence""","""Jane Remover""","""""",null,null
1740762231,2025-02-28 17:03:51,"""hot problems""","""food house""","""""",null,null
1740764659,2025-02-28 17:44:19,"""Digital Girl""","""DJ Suzy""","""""",null,null


In [ ]:
# df.with_row_index("idx").filter(pl.col("artist_name") == "Iglooghost")

idx,date_played_unix,track_played_utc,track_name,artist_name,track_mbid
u32,i64,datetime[μs],str,str,str
430,1611730525,2021-01-27 06:55:25,"""Amu""","""Iglooghost""",""""""
17681,1761054834,2025-10-21 13:53:54,"""Blue Hum""","""Iglooghost""","""72596ba2-974b-47d4-b51b-988116…"
17682,1761054966,2025-10-21 13:56:06,"""New Species""","""Iglooghost""","""f85163a7-e982-4eb6-b8e2-e633a9…"
17683,1761064405,2025-10-21 16:33:25,"""Alloy Flea""","""Iglooghost""","""60791d72-ff67-4df3-a453-f8cfa3…"
17684,1761079598,2025-10-21 20:46:38,"""Spawn01 ft Cyst""","""Iglooghost""","""2e43febd-2bec-437f-bbe6-14f195…"
17685,1761123638,2025-10-22 09:00:38,"""Coral Mimic""","""Iglooghost""","""4e08fdc9-ce2d-4e48-afc3-299a91…"
17722,1761330827,2025-10-24 18:33:47,"""Xcavator.03""","""Iglooghost""","""0403cea1-b307-42bb-9025-f6bbb2…"
17724,1761381408,2025-10-25 08:36:48,"""flux•Cocoon""","""Iglooghost""","""8872a941-76d4-487c-9c26-3782e6…"


In [38]:
pl.read_parquet(
        Path("../data/spotify/2017-2021 spotify data.parquet"))

track_played_utc,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],str,str,str,i64,str
2017-12-17 10:31:16,"""Like A G6""","""Far East Movement""","""Free Wired""",1513506676,"""spotify:track:5AyL2kgLtTWEu3qO…"
2017-12-17 10:34:47,"""Rocketeer""","""Far East Movement""","""Free Wired""",1513506887,"""spotify:track:3ZuiIFm9nRnkRz8A…"
2017-12-17 10:38:45,"""Live My Life""","""Far East Movement""","""Dirty Bass""",1513507125,"""spotify:track:6dpKQiQzZE2r9rZV…"
2017-12-17 10:41:56,"""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…",1513507316,"""spotify:track:00rbUJGbcjw3FW5c…"
2017-12-17 10:43:01,"""Turn Up The Love""","""Far East Movement""","""Dirty Bass""",1513507381,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…
2021-05-20 06:11:42,"""Xcxoplex (with Charli XCX)""","""A. G. Cook""","""Xcxoplex (with Charli XCX)""",1621491102,"""spotify:track:3Qtc8RGzuzbDGBpZ…"
2021-05-20 06:15:39,"""Miss The Rage (feat. Playboi C…","""Trippie Redd""","""Miss The Rage (feat. Playboi C…",1621491339,"""spotify:track:2BITQ360Knh6qNAO…"
2021-05-20 06:18:03,"""The Kiss Of Venus (Dominic Fik…","""Paul McCartney""","""McCartney III Imagined""",1621491483,"""spotify:track:28kOGtTZzbfQ8fMm…"


In [ ]:
import polars as pl
def get_artist_table():
    lastfm_df = pl.read_parquet(
    Path("../data/lastfm/lastfm-listening-2021-2026march.parquet"), columns=[
    "date_played_unix", "track_played_utc", "artist_name", "artist_mbid", ]
    )
    spotify_df = pl.read_parquet(
        Path("../data/spotify/2017-2021 spotify data.parquet"),
        columns=["date_played_unix", "track_played_utc", "artist_name"],
    )
    df = pl.concat([ lastfm_df, spotify_df ], how="diagonal").sort(pl.col("date_played_unix"), descending=False)
    df = ( 
        df
        .filter(pl.col("track_played_utc").dt.year() >= 2021)
        .unique(subset=["artist_name"], keep="first")
        .sort("track_played_utc", descending=False)
        .drop_nulls(pl.col("artist_name"))
    )

    return df

get_artist_table()

date_played_unix,track_played_utc,artist_name,artist_mbid
i64,datetime[μs],str,str
1609460125,2021-01-01 00:15:25,"""The Kid LAROI""",null
1609460335,2021-01-01 00:18:55,"""TV Girl""",null
1609504835,2021-01-01 12:40:35,"""Her's""",null
1609505050,2021-01-01 12:44:10,"""Playboi Carti""",null
1609505246,2021-01-01 12:47:26,"""Pity Party (Girls Club)""",null
…,…,…,…
1774168414,2026-03-22 08:33:34,"""Dagmar Zuniga""",""""""
1774187066,2026-03-22 13:44:26,"""Tokyo Tea Room""","""f836a8dd-6b20-4d95-9e0f-5b758f…"
1774187541,2026-03-22 13:52:21,"""Michael Seyer""","""01d243c3-ff03-4470-ae10-a23298…"
